# TPC-H Complex Queries

**Dataset:** `samples.tpch (all tables)`

**Difficulty:** Hard

**Topics:** TPC-H benchmark queries adapted, multi-join, subqueries, window, aggregation

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
lineitem = spark.table("samples.tpch.lineitem")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
region   = spark.table("samples.tpch.region")
part     = spark.table("samples.tpch.part")
supplier = spark.table("samples.tpch.supplier")
partsupp = spark.table("samples.tpch.partsupp")

## Problem 1

**Pricing Summary (TPC-H Q1 variant).**

For all line items shipped on or before `1998-09-01`, group by `l_returnflag` and `l_linestatus`. Compute:
- `sum_qty` = sum of l_quantity
- `sum_base_price` = sum of l_extendedprice
- `sum_disc_price` = sum of l_extendedprice * (1 - l_discount)
- `sum_charge` = sum of l_extendedprice * (1 - l_discount) * (1 + l_tax)
- `avg_qty`, `avg_price` (avg extendedprice), `avg_disc`, `count_order`

Business context: This is the foundational pricing summary query used by finance to reconcile revenue recognition — it categorises all shipped goods by their fulfilment and return status.

**Expected output columns:** `l_returnflag`, `l_linestatus`, `sum_qty`, `sum_base_price`, `sum_disc_price`, `sum_charge`, `avg_qty`, `avg_price`, `avg_disc`, `count_order`

In [ ]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None — did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for col in ['l_returnflag', 'l_linestatus', 'sum_qty', 'sum_base_price',
            'sum_disc_price', 'sum_charge', 'avg_qty', 'avg_price', 'avg_disc', 'count_order']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 10, f"Expected exactly 10 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_1.agg(F.min('sum_qty')).collect()[0][0] > 0, "sum_qty must be positive"
assert result_1.agg(F.min('count_order')).collect()[0][0] > 0, "count_order must be positive"
# sum_disc_price <= sum_base_price (discount reduces price)
check = result_1.filter(F.col('sum_disc_price') > F.col('sum_base_price')).count()
assert check == 0, "sum_disc_price should be <= sum_base_price (discount applied)"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

**Shipping Priority (TPC-H Q3 variant).**

Find unshipped orders with the highest revenue. Join:
- `customer` (filter `c_mktsegment = 'BUILDING'`)
- `orders` (filter `o_orderdate < '1995-03-15'`)
- `lineitem` (filter `l_shipdate > '1995-03-15'`)

Compute `revenue = sum(l_extendedprice * (1 - l_discount))` per order key. Return the top 10 orders by revenue descending.

Business context: Shipping priority analysis helps the logistics team triage unfulfilled high-revenue orders from the most profitable customer segment — ensuring the highest-value orders are expedited first.

**Expected output columns:** `l_orderkey`, `revenue`, `o_orderdate`, `o_shippriority`

In [ ]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None — did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for col in ['l_orderkey', 'revenue', 'o_orderdate', 'o_shippriority']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt == 10, f"Expected exactly 10 rows (top 10 orders), got {cnt}"
assert result_2.agg(F.min('revenue')).collect()[0][0] > 0, "revenue must be positive"
# verify descending order
revenues = [r['revenue'] for r in result_2.orderBy('revenue', ascending=False).collect()]
assert revenues == sorted(revenues, reverse=True), "Results should be sorted by revenue descending"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

**Order Priority Checking (TPC-H Q4 variant).**

Count orders placed in Q3 1993 (July–September 1993) where at least one line item was received late (`l_receiptdate > l_commitdate`). Group by `o_orderpriority` and sort ascending.

Business context: The operations review board uses this query to assess whether high-priority orders receive preferential treatment in the supply chain or whether priority labels are meaningless in practice.


**Expected output columns:** `o_orderpriority`, `order_count`

In [ ]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None — did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'o_orderpriority' in cols, "Missing column: o_orderpriority"
assert 'order_count' in cols, "Missing column: order_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 5, f"Expected at most 5 priority levels, got {cnt}"
assert result_3.agg(F.min('order_count')).collect()[0][0] > 0, "order_count must be positive"
priorities = [r['o_orderpriority'] for r in result_3.select('o_orderpriority').collect()]
assert len(priorities) == len(set(priorities)), "o_orderpriority should be unique per row"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

**Local Supplier Volume (TPC-H Q5 variant).**

For all orders placed in 1994, compute revenue where the customer and the supplier are in the same nation. Join customer → nation, orders, lineitem, supplier → nation, filtering where `c_nationkey = s_nationkey`. Revenue = `sum(l_extendedprice * (1 - l_discount))`. Group by nation name, sort descending.

Business context: Understanding local supply-chain volume helps the procurement team identify which countries have strong domestic supply loops and where international sourcing is dominating, informing nearshoring strategy.


**Expected output columns:** `n_name`, `revenue`

In [ ]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None — did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'n_name' in cols, "Missing column: n_name"
assert 'revenue' in cols, "Missing column: revenue"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_4.agg(F.min('revenue')).collect()[0][0] > 0, "revenue must be positive"
nation_names = result_4.select('n_name').distinct().count()
assert nation_names == cnt, "Each n_name should appear only once"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

**Revenue Change Forecast (TPC-H Q6 variant).**

Compute the potential revenue increase if all discounts between 0.05 and 0.07 on line items with quantity < 24 were eliminated, for ship dates in 1994. `revenue_change = sum(l_extendedprice * l_discount)`. Return a single-row result.

Business context: The CFO's office wants to understand the total revenue uplift from eliminating small promotional discounts on low-volume orders, to assess whether a blanket discount-removal policy is financially worthwhile.

**Expected output columns:** `revenue_change`

In [ ]:
# Problem 5 - write your solution here
# Assign your results to: result_5, rc

result_5 = None  # replace this
rc = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None — did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'revenue_change' in cols, "Missing column: revenue_change"
cnt = result_5.count()
assert cnt == 1, f"Expected exactly 1 row (single aggregated value), got {cnt}"
rc = result_5.collect()[0]['revenue_change']
assert rc is not None, "revenue_change must not be null"
assert float(rc) > 0, f"revenue_change must be positive, got {rc}"
print(f"Problem 5 passed ✓  (revenue_change = {float(rc):,.2f})")

## Problem 6

**Volume Shipping (TPC-H Q7 variant).**

Find gross discounted revenue for goods shipped between France and Germany (in either direction — supplier in one country, customer in the other) in 1995–1996. Join nation twice (for supplier and customer), filter `n_name IN ('FRANCE', 'GERMANY')` on both sides, and compute `revenue = sum(l_extendedprice * (1 - l_discount))` per supp_nation, cust_nation, and year of shipdate.

Business context: Cross-border trade volume analysis between France and Germany helps the European logistics division plan cross-border capacity and pricing for the following fiscal year.


**Expected output columns:** `supp_nation`, `cust_nation`, `l_year`, `revenue`

In [ ]:
# Problem 6 - write your solution here
# Assign your result to: result_6

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None — did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for col in ['supp_nation', 'cust_nation', 'l_year', 'revenue']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
nations = result_6.select('supp_nation').distinct().collect()
nation_set = {r['supp_nation'] for r in nations}
assert nation_set.issubset({'FRANCE', 'GERMANY'}), f"Unexpected nations: {nation_set}"
assert result_6.agg(F.min('revenue')).collect()[0][0] > 0, "revenue must be positive"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

**Market Share Analysis.**

For each region compute what percentage of total revenue (across all regions) comes from that region, broken down by order year. Revenue = `sum(l_extendedprice * (1 - l_discount))`. Join: orders → customer → nation → region → lineitem.

Business context: Regional revenue share trends help the executive team spot emerging growth markets and declining regions, informing where to invest in sales headcount and marketing spend.


**Expected output columns:** `r_name`, `o_year`, `regional_revenue`, `total_revenue`, `market_share_pct`

In [ ]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None — did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
for col in ['r_name', 'o_year', 'regional_revenue', 'total_revenue', 'market_share_pct']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# For each year, all region shares should sum to ~100
year_sums = result_7.groupBy('o_year').agg(F.sum('market_share_pct').alias('total_pct'))
bad_years = year_sums.filter((F.col('total_pct') < 99.0) | (F.col('total_pct') > 101.0)).count()
assert bad_years == 0, f"Found {bad_years} years where market shares don't sum to ~100%"
assert result_7.agg(F.min('regional_revenue')).collect()[0][0] > 0, "regional_revenue must be positive"
print(f"Problem 7 passed ✓  ({cnt} rows)")

## Problem 8

**Supplier Reliability.**

For each supplier compute: total parts supplied (distinct part count from partsupp), average supply cost, number of parts where available quantity (`ps_availqty`) < 100 (low stock count), and `low_stock_pct = low_stock_count / total_parts * 100`. Join supplier → nation → region to include geographic context.

Business context: Procurement risk management requires knowing which suppliers are running critically low stock across their part portfolios — especially those in regions with long lead times.

**Expected output columns:** `s_name`, `n_name`, `r_name`, `total_parts`, `avg_supply_cost`, `low_stock_count`, `low_stock_pct`

In [ ]:
# Problem 8 - write your solution here
# Assign your result to: result_8

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None — did you assign your DataFrame?"
assert hasattr(result_8, 'columns'), "result_8 must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
for col in ['s_name', 'n_name', 'r_name', 'total_parts',
            'avg_supply_cost', 'low_stock_count', 'low_stock_pct']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 7, f"Expected exactly 7 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_8.agg(F.min('total_parts')).collect()[0][0] > 0, "total_parts must be positive"
assert result_8.agg(F.min('low_stock_pct')).collect()[0][0] >= 0, "low_stock_pct must be non-negative"
assert result_8.agg(F.max('low_stock_pct')).collect()[0][0] <= 100, "low_stock_pct cannot exceed 100"
print(f"Problem 8 passed ✓  ({cnt} rows)")